# 02 — Bronze Ingestion

**Week:** 4

Goal: Preserve raw records with ingestion metadata and reconcile row counts.


In [0]:


from pyspark.sql.functions import current_timestamp, lit

raw_path = "/Volumes/ship_track/default/ship_track_volume/carriers.csv"

bronze_df = (
    spark.read.option("header", True).option("inferSchema", True).csv(raw_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("ShipTrack"))
    .withColumn("source_file_name", lit("carriers.csv"))
    .withColumn("batch_id", lit("batch_001"))
)

display(bronze_df.limit(10))

source_record_id,carrier_id,carrier_name,transport_mode,primary_service_level,active_status,effective_start_date,effective_end_date,ingestion_timestamp,source_system,source_file_name,batch_id
CARREC00001,C001,Aster Freight,ROAD,STANDARD,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00002,C002,BlueArc Logistics,ROAD,EXPRESS,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00003,C003,Cobalt Cargo,AIR,PRIORITY,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00004,C004,DeltaLine Transport,RAIL,STANDARD,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00005,C005,EmberRoute Express,AIR,EXPRESS,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00006,C006,Falcon Surface,ROAD,PRIORITY,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00007,C007,GreenSpan Carriers,ROAD,STANDARD,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00008,C008,HarborLink Cargo,RAIL,EXPRESS,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00009,C009,IndigoTrail Logistics,AIR,STANDARD,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001
CARREC00010,C010,Legacy Carrier,ROAD,STANDARD,INACTIVE,2023-01-01,2025-12-31,2026-07-31T09:40:30.824Z,ShipTrack,carriers.csv,batch_001


In [0]:
bronze_table = "bronze_carriers"

bronze_df.write.mode("overwrite").saveAsTable(bronze_table)

print("Bronze table created:", bronze_table)
print("Bronze row count:", spark.table(bronze_table).count())


Bronze table created: bronze_carriers
Bronze row count: 10


In [0]:
%sql
SELECT source_system, source_file_name, COUNT(*) AS record_count
FROM bronze_carriers
GROUP BY source_system, source_file_name


source_system,source_file_name,record_count
ShipTrack,carriers.csv,10


In [0]:
raw_path = "/Volumes/ship_track/default/ship_track_volume/exceptions.csv"

bronze_df = (
    spark.read.option("header", True).option("inferSchema", True).csv(raw_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("ShipTrack"))
    .withColumn("source_file_name", lit("exceptions.csv"))
    .withColumn("batch_id", lit("batch_001"))
)

display(bronze_df.limit(10))


source_record_id,exception_id,shipment_id,scan_id,exception_type,exception_time,hub_id,severity,resolution_status,resolution_time,source_system,run_id,ingestion_timestamp,source_file_name,batch_id
EXCREC00000001,EXC00000001,SHP00000009,SCN000000060,MISSED_CONNECTION,2026-01-06T03:23:20.000Z,H005,MEDIUM,RESOLVED,2026-01-06T06:46:05.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000002,EXC00000002,SHP00000010,SCN000000069,CAPACITY,2026-06-02T06:05:59.000Z,H010,HIGH,RESOLVED,2026-06-02T21:47:47.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000003,EXC00000003,SHP00000020,SCN000000146,PACKAGE_DAMAGED,2026-05-25T15:36:39.000Z,H001,LOW,RESOLVED,2026-05-26T02:45:18.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000004,EXC00000004,SHP00000021,SCN000000155,CAPACITY,2026-01-05T18:36:03.000Z,H002,MEDIUM,RESOLVED,2026-01-05T23:16:39.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000005,EXC00000005,SHP00000023,SCN000000167,ADDRESS_ISSUE,2026-04-14T17:30:09.000Z,H013,MEDIUM,RESOLVED,2026-04-14T23:30:36.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000006,EXC00000006,SHP00000030,SCN000000219,WEATHER,2026-01-11T20:55:19.000Z,H011,MEDIUM,RESOLVED,2026-01-12T05:06:07.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000007,EXC00000007,SHP00000038,SCN000000267,WEATHER,2026-06-09T09:01:11.000Z,H009,LOW,OPEN,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000008,EXC00000008,SHP00000038,SCN000000270,ADDRESS_ISSUE,2026-06-10T21:53:27.000Z,H005,HIGH,CLOSED,2026-06-10T21:53:27.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000009,EXC00000009,SHP00000041,SCN000000291,ADDRESS_ISSUE,2026-03-24T09:44:35.000Z,H008,HIGH,RESOLVED,2026-03-25T03:19:22.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001
EXCREC00000010,EXC00000010,SHP00000046,SCN000000324,ADDRESS_ISSUE,2026-06-03T18:47:55.000Z,H013,MEDIUM,RESOLVED,2026-06-04T10:27:20.000Z,ShipTrack,P10_STAGE1_R1,2026-07-31T09:41:53.642Z,exceptions.csv,batch_001


In [0]:
bronze_table = "bronze_exceptions"

bronze_df.write.mode("overwrite").saveAsTable(bronze_table)

print("Bronze table created:", bronze_table)
print("Bronze row count:", spark.table(bronze_table).count())


Bronze table created: bronze_exceptions
Bronze row count: 18352


In [0]:

%sql
SELECT source_system, source_file_name, COUNT(*) AS record_count
FROM bronze_exceptions
GROUP BY source_system, source_file_name

source_system,source_file_name,record_count
ShipTrack,exceptions.csv,18352


In [0]:
raw_path = "/Volumes/ship_track/default/ship_track_volume/routes.csv"

bronze_df = (
    spark.read.option("header", True).option("inferSchema", True).csv(raw_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("ShipTrack"))
    .withColumn("source_file_name", lit("routes.csv"))
    .withColumn("batch_id", lit("batch_001"))
)

display(bronze_df.limit(10))

source_record_id,route_id,origin_hub_id,destination_hub_id,service_level,transport_mode,distance_km,expected_transit_hours,route_band,active_status,effective_start_date,effective_end_date,ingestion_timestamp,source_system,source_file_name,batch_id
RTREC00001,R001,H002,H010,EXPRESS,ROAD,1011.0,27.5,MEDIUM,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00002,R002,H013,H009,STANDARD,AIR,1325.0,16.0,LONG,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00003,R003,H003,H009,EXPRESS,ROAD,1002.0,29.6,MEDIUM,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00004,R004,H005,H013,PRIORITY,RAIL,2129.0,40.7,LONG,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00005,R005,H006,H011,STANDARD,AIR,2266.0,19.2,LONG,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00006,R006,H013,H010,STANDARD,ROAD,653.0,26.1,MEDIUM,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00007,R007,H008,H011,STANDARD,ROAD,2149.0,57.5,LONG,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00008,R008,H012,H007,EXPRESS,ROAD,1971.0,49.2,LONG,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00009,R009,H010,H003,STANDARD,ROAD,1906.0,54.3,LONG,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001
RTREC00010,R010,H009,H001,PRIORITY,RAIL,764.0,19.2,MEDIUM,ACTIVE,2024-01-01,2099-12-31,2026-07-31T09:44:56.734Z,ShipTrack,routes.csv,batch_001


In [0]:
bronze_table = "bronze_routes"

bronze_df.write.mode("overwrite").saveAsTable(bronze_table)

print("Bronze table created:", bronze_table)
print("Bronze row count:", spark.table(bronze_table).count())

Bronze table created: bronze_routes
Bronze row count: 107


In [0]:
%sql
SELECT source_system, source_file_name, COUNT(*) AS record_count
FROM bronze_routes
GROUP BY source_system, source_file_name

source_system,source_file_name,record_count
ShipTrack,routes.csv,107


In [0]:
raw_path = "/Volumes/ship_track/default/ship_track_volume/scan_events.csv"

bronze_df = (
    spark.read.option("header", True).option("inferSchema", True).csv(raw_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("ShipTrack"))
    .withColumn("source_file_name", lit("scan_events.csv"))
    .withColumn("batch_id", lit("batch_001"))
)

display(bronze_df.limit(10))

source_record_id,scan_id,shipment_id,event_sequence_no,event_type,shipment_status,event_time,hub_id,carrier_id,route_id,exception_type,package_condition,status_reason,source_system,run_id,ingestion_timestamp,source_file_name,batch_id
SCNREC000000001,SCN000000001,SHP00000001,1,CREATED,CREATED,2026-05-06T09:30:00.000Z,H013,C006,R065,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000002,SCN000000002,SHP00000001,2,PICKED_UP,PICKED_UP,2026-05-06T18:59:30.000Z,H013,C006,R065,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000003,SCN000000003,SHP00000001,3,DEPARTED_HUB,IN_TRANSIT,2026-05-07T02:28:47.000Z,H013,C006,R065,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000004,SCN000000004,SHP00000001,4,ARRIVED_HUB,AT_HUB,2026-05-07T14:57:35.000Z,H006,C006,R065,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000005,SCN000000005,SHP00000001,5,DEPARTED_HUB,IN_TRANSIT,2026-05-08T00:06:42.000Z,H006,C006,R065,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000006,SCN000000006,SHP00000001,6,ARRIVED_HUB,AT_HUB,2026-05-08T06:46:04.000Z,H011,C006,R065,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000007,SCN000000007,SHP00000001,7,STATUS_UPDATE,OUT_FOR_DELIVERY,2026-05-08T10:15:50.000Z,H011,C006,R065,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000008,SCN000000008,SHP00000001,8,DELIVERY_CONFIRMATION,DELIVERED,2026-05-08T13:15:50.000Z,H011,C006,R065,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000009,SCN000000009,SHP00000002,1,CREATED,CREATED,2026-01-30T22:44:00.000Z,H003,C008,R036,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001
SCNREC000000010,SCN000000010,SHP00000002,2,PICKED_UP,PICKED_UP,2026-01-31T07:36:40.000Z,H003,C008,R036,null,GOOD,null,ShipTrack,P10_STAGE1_R1,2026-07-31T09:45:11.428Z,scan_events.csv,batch_001


In [0]:
bronze_table = "bronze_scan_events"

bronze_df.write.mode("overwrite").saveAsTable(bronze_table)

print("Bronze table created:", bronze_table)
print("Bronze row count:", spark.table(bronze_table).count())

Bronze table created: bronze_scan_events
Bronze row count: 710666


In [0]:
%sql
SELECT source_system, source_file_name, COUNT(*) AS record_count
FROM bronze_scan_events
GROUP BY source_system, source_file_name

source_system,source_file_name,record_count
ShipTrack,scan_events.csv,710666


In [0]:
import pyarrow.parquet as pq

raw_path = "/Volumes/ship_track/default/ship_track_volume/shipments.parquet"

# Read parquet with PyArrow to handle nanosecond timestamps
pa_table = pq.read_table(raw_path)
bronze_df = (
    spark.createDataFrame(pa_table.to_pandas())
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("ShipTrack"))
    .withColumn("source_file_name", lit("shipments.parquet"))
    .withColumn("batch_id", lit("batch_001"))
)

display(bronze_df.limit(10))

source_record_id,shipment_id,order_reference,route_id,origin_hub_id,destination_hub_id,carrier_id,service_level,transport_mode,booking_ts,pickup_ts,promised_delivery_ts,actual_delivery_ts,return_completed_ts,latest_status,delivery_outcome,status_reason,package_weight_kg,package_count,freight_amount_inr,attempt_count,exception_flag,source_system,run_id,ingestion_timestamp,source_file_name,batch_id
SHPREC00000001,SHP00000001,ORD000000001,R065,H013,H011,C006,STANDARD,ROAD,2026-05-06T09:30:00.000Z,2026-05-06T18:59:30.855Z,2026-05-09T00:35:30.855Z,2026-05-08T13:15:50.055Z,null,DELIVERED,DELIVERED,,8.16,3,1072.54,1,false,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000002,SHP00000002,ORD000000002,R036,H003,H005,C008,EXPRESS,RAIL,2026-01-30T22:44:00.000Z,2026-01-31T07:36:40.199Z,2026-02-02T07:06:40.199Z,2026-02-02T01:47:28.199Z,null,DELIVERED,DELIVERED,,3.47,4,1698.96,1,false,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000003,SHP00000003,ORD000000003,R094,H009,H004,C009,EXPRESS,AIR,2026-05-09T03:22:00.000Z,2026-05-09T05:37:03.499Z,2026-05-09T22:43:03.499Z,2026-05-09T19:49:46.699Z,null,DELIVERED,DELIVERED,,9.48,1,1724.68,2,false,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000004,SHP00000004,ORD000000004,R036,H003,H005,C008,EXPRESS,RAIL,2026-06-19T21:16:00.000Z,2026-06-20T13:50:08.442Z,2026-06-22T13:20:08.442Z,2026-06-22T22:59:51.551Z,null,DELIVERED,DELIVERED,,4.33,2,1638.35,1,false,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000005,SHP00000005,ORD000000005,R075,H006,H012,C008,PRIORITY,RAIL,2026-06-23T00:50:00.000Z,2026-06-23T15:23:05.328Z,2026-06-24T19:17:05.328Z,2026-06-25T02:51:05.568Z,null,DELIVERED,DELIVERED,,2.49,1,1316.57,1,false,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000006,SHP00000006,ORD000000006,R046,H001,H008,C006,STANDARD,ROAD,2026-05-21T20:33:00.000Z,2026-05-22T03:37:15.624Z,2026-05-23T23:43:15.624Z,null,null,OUT_FOR_DELIVERY,OUT_FOR_DELIVERY,,14.87,2,873.48,0,false,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000007,SHP00000007,ORD000000007,R015,H012,H002,C006,PRIORITY,ROAD,2026-02-07T15:46:00.000Z,2026-02-08T00:24:05.958Z,2026-02-09T08:42:05.958Z,2026-02-09T07:21:27.558Z,null,DELIVERED,DELIVERED,,4.98,1,1168.07,1,false,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000008,SHP00000008,ORD000000008,R079,H004,H013,C007,EXPRESS,ROAD,2026-05-16T12:23:00.000Z,null,2026-05-17T22:11:00.000Z,null,null,CANCELLED,CANCELLED,CAPACITY_REPLAN,4.31,2,868.44,0,false,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000009,SHP00000009,ORD000000009,R039,H011,H012,C007,EXPRESS,ROAD,2026-01-04T19:40:00.000Z,2026-01-05T11:19:37.802Z,2026-01-06T22:49:37.802Z,2026-01-06T18:28:01.802Z,null,DELIVERED,DELIVERED,,6.4,3,1131.31,1,true,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001
SHPREC00000010,SHP00000010,ORD000000010,R063,H009,H004,C007,STANDARD,ROAD,2026-06-01T14:04:00.000Z,2026-06-01T17:49:33.099Z,2026-06-03T06:49:33.099Z,2026-06-03T00:36:16.760Z,null,DELIVERED,DELIVERED,,9.4,3,719.21,1,true,ShipTrack,P10_STAGE1_R1,2026-07-31T09:47:25.879Z,shipments.parquet,batch_001


In [0]:
bronze_table = "bronze_shipments"

bronze_df.write.mode("overwrite").saveAsTable(bronze_table)

print("Bronze table created:", bronze_table)
print("Bronze row count:", spark.table(bronze_table).count())

Bronze table created: bronze_shipments
Bronze row count: 100020


In [0]:
%sql
SELECT source_system, source_file_name, COUNT(*) AS record_count
FROM bronze_shipments
GROUP BY source_system, source_file_name

source_system,source_file_name,record_count
ShipTrack,shipments.parquet,100020


In [0]:
raw_path = "/Volumes/ship_track/default/ship_track_volume/hubs.json"

bronze_df = (
    spark.read.option("multiline", True).json(raw_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("ShipTrack"))
    .withColumn("source_file_name", lit("hubs.json"))
    .withColumn("batch_id", lit("batch_001"))
)

display(bronze_df.limit(10))

active_status,capacity_band,city,effective_end_date,effective_start_date,hub_id,hub_name,hub_type,region,source_record_id,ingestion_timestamp,source_system,source_file_name,batch_id
ACTIVE,HIGH,Hyderabad,2099-12-31,2024-01-01,H001,Deccan Gateway Hub,GATEWAY,South,HUBREC00001,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,HIGH,Bengaluru,2099-12-31,2024-01-01,H002,Cauvery North Hub,REGIONAL,South,HUBREC00002,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,HIGH,Chennai,2099-12-31,2024-01-01,H003,Coromandel Transit Hub,REGIONAL,South,HUBREC00003,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,HIGH,Mumbai,2099-12-31,2024-01-01,H004,Konkan Gateway Hub,GATEWAY,West,HUBREC00004,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,MEDIUM,Ahmedabad,2099-12-31,2024-01-01,H005,Sabarmati Regional Hub,REGIONAL,West,HUBREC00005,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,HIGH,Delhi,2099-12-31,2024-01-01,H006,Yamuna Gateway Hub,GATEWAY,North,HUBREC00006,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,MEDIUM,Lucknow,2099-12-31,2024-01-01,H007,Ganga Regional Hub,REGIONAL,North,HUBREC00007,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,HIGH,Kolkata,2099-12-31,2024-01-01,H008,Hooghly Gateway Hub,GATEWAY,East,HUBREC00008,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,MEDIUM,Bhubaneswar,2099-12-31,2024-01-01,H009,Kalinga Regional Hub,REGIONAL,East,HUBREC00009,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001
ACTIVE,MEDIUM,Indore,2099-12-31,2024-01-01,H010,Narmada Transit Hub,TRANSIT,Central,HUBREC00010,2026-07-31T09:48:16.993Z,ShipTrack,hubs.json,batch_001


In [0]:
bronze_table = "bronze_hubs"

bronze_df.write.mode("overwrite").saveAsTable(bronze_table)

print("Bronze table created:", bronze_table)
print("Bronze row count:", spark.table(bronze_table).count())

Bronze table created: bronze_hubs
Bronze row count: 14


In [0]:
%sql
SELECT source_system, source_file_name, COUNT(*) AS record_count
FROM bronze_hubs
GROUP BY source_system, source_file_name

source_system,source_file_name,record_count
ShipTrack,hubs.json,14


In [0]:
%sql
SHOW TABLES;


database,tableName,isTemporary
default,bronze_carriers,false
default,bronze_exceptions,false
default,bronze_hubs,false
default,bronze_routes,false
default,bronze_scan_events,false
default,bronze_shipments,false
default,shiptrack_week03_bronze_demo_shipments,false
default,shiptrack_week03_lineage_demo_view,false


In [0]:
import pyarrow.parquet as pq

datasets = [
    ("carriers.csv", "bronze_carriers", "csv"),
    ("exceptions.csv", "bronze_exceptions", "csv"),
    ("routes.csv", "bronze_routes", "csv"),
    ("scan_events.csv", "bronze_scan_events", "csv"),
    ("shipments.parquet", "bronze_shipments", "parquet"),
    ("hubs.json", "bronze_hubs", "json")
]

for file, table, fmt in datasets:
    path = f"/Volumes/ship_track/default/ship_track_volume/{file}"

    if fmt == "csv":
        raw = spark.read.option("header", True).csv(path).count()
    elif fmt == "json":
        raw = spark.read.option("multiline", True).json(path).count()
    else:
        pa_table = pq.read_table(path)
        raw = len(pa_table)

    bronze = spark.table(table).count()

    print(f"{table:25} Raw={raw:<8} Bronze={bronze}")

bronze_carriers           Raw=10       Bronze=10
bronze_exceptions         Raw=18352    Bronze=18352
bronze_routes             Raw=107      Bronze=107
bronze_scan_events        Raw=710666   Bronze=710666
bronze_shipments          Raw=100020   Bronze=100020
bronze_hubs               Raw=14       Bronze=14


## Acceptance Criteria

- Bronze table exists
- Metadata columns exist
- Raw count and Bronze count reconcile
- Screenshot saved in `screenshots/`
